# Handoffs Pattern (에이전트 간 핸드오프 패턴)

**핸드오프 패턴(Handoffs Pattern)** 은 단일 에이전트가 여러 상태(state)를 거치며 작업을 수행하는 패턴입니다.
각 상태마다 다른 프롬프트와 도구를 사용하여, 복잡한 워크플로우를 단일 에이전트로 구현할 수 있습니다.

이 튜토리얼에서는 고객 지원(Customer Support) 시나리오를 구현합니다:
- 보증 상태 확인 → 이슈 분류 → 해결책 제공

**참고**: [LangChain 공식 문서 - Handoffs Customer Support](https://docs.langchain.com/oss/python/langchain/multi-agent/handoffs-customer-support)

In [ ]:
# LangSmith 추적 (선택적)

## 1. 상태 스키마 정의

워크플로우의 상태를 정의합니다. 각 단계와 수집된 정보를 저장합니다.
```
Agent Workflow  
   ↓  
State(State 객체)  
   ↓
각 Step 노드가 State 읽고 수정
   ↓
그래프가 다음 Step 결정
```

In [ ]:
# 가능한 워크플로우 단계
# 보증 상태
# 이슈 유형
class SupportState(AgentState):

## 2. 상태 관리 도구 정의

각 단계에서 상태를 업데이트하는 도구를 정의합니다.

In [ ]:
def record_warranty_status(
            # Tool 실행 결과를 메시지 히스토리에 추가
                    # Tool 호출과 메시지를 연결하기 위한 ID
            # 고객 보증 상태를 State 에 저장
            # 다음 워크플로우 단계 지정
def record_issue_type(
            # Tool 실행 결과 메시지 기록
            # 문제 유형 상태 저장
            # 다음 단계 설정
def escalate_to_human(reason: str) -> str:
    # 실제 서비스에서는 외부 시스템 연동 로직이 들어갈 위치
def provide_solution(solution: str) -> str:

## 3. 단계별 프롬프트 정의

각 단계마다 다른 시스템 프롬프트를 사용합니다.

In [ ]:
# 보증 수집 단계 프롬프트
# 이슈 분류 단계 프롬프트
# 해결책 제공 단계 프롬프트

## 4. 단계별 설정(STEP_CONFIG) 및 wrap_model_call 미들웨어

최신 문서: 현재 상태(current_step)에 따라 프롬프트와 도구를 request.override로 주입합니다.

In [ ]:
# ------------------------------------------------------------
# 단계별 Agent 설정 정보
# ------------------------------------------------------------
# 각 워크플로우 단계마다
#   1. 사용할 시스템 프롬프트
#   2. 허용할 Tool 목록
#   3. 해당 단계에 진입하기 위한 필수 상태(State)
# 를 정의하는 설정 테이블
    # -----------------------------
    # 보증 상태 수집 단계
    # -----------------------------
    # -----------------------------
    # 이슈 분류 단계
    # -----------------------------
    # -----------------------------
    # 문제 해결 단계
    # -----------------------------
# ------------------------------------------------------------
# Model Call Middleware 정의
# ------------------------------------------------------------
# wrap_model_call:
#   → LLM 호출 직전에 개입하여
#     - 시스템 프롬프트 변경
#     - Tool 목록 변경
#     - 요청 파라미터 수정
#   등을 수행할 수 있는 미들웨어 데코레이터
def apply_step_config(
    # ------------------------------------------------------------
    # 현재 단계 확인
    # ------------------------------------------------------------
    # state 에 current_step 값이 없으면 기본 단계는 warranty_collector
    # 해당 단계 설정 정보 조회
    # ------------------------------------------------------------
    # 단계 진입 조건 검증
    # ------------------------------------------------------------
    # 특정 단계는 반드시 선행 상태가 있어야 진입 가능
    # 예:
    # issue_classifier → warranty_status 필요
    # resolution_specialist → warranty_status + issue_type 필요
    # ------------------------------------------------------------
    # 시스템 프롬프트 동적 생성
    # ------------------------------------------------------------
    # 현재 상태 값을 프롬프트에 삽입하여
    # LLM 이 현재 상황을 이해하도록 만듦
        # 보증 상태
        # 문제 유형
    # 단계별 프롬프트 템플릿에 상태 값 삽입
    # ------------------------------------------------------------
    # ModelRequest 수정
    # ------------------------------------------------------------
    # override():
    #   기존 요청 객체를 복사하면서
    #   특정 필드만 변경하는 불변 객체 패턴
    # 수정된 요청을 실제 LLM 호출 핸들러로 전달

## 5. 에이전트 생성

state_schema와 apply_step_config 미들웨어로 단계별 프롬프트·도구를 적용합니다.

In [ ]:
# Agent 에서 사용할 모든 Tool 목록 정의
# Agent 생성
    # Agent 실행 중 상태를 구조적으로 관리하기 위한 타입 정의
    # --------------------------------------------------------
    # Middleware 설정
    # --------------------------------------------------------
    # apply_step_config:
    #   - 현재 단계 확인
    #   - 시스템 프롬프트 변경
    #   - Tool 사용 제한

## 7. 멀티 턴 대화 실행

여러 턴에 걸쳐 상태 전환을 확인합니다.

In [ ]:
# 스레드 ID 생성
#   → Agent 대화 세션 식별자
#   → Checkpointer 가 상태를 저장할 때 사용됨
#   → 같은 thread_id 를 사용하면 이전 상태가 이어짐
# Agent 실행
# Agent 응답 메시지 출력
    # pretty_print 메서드가 있으면 보기 좋게 출력
# 현재 Workflow 단계 출력
# current_step:
#   → Agent 상태(State)에 저장된 현재 단계

### 턴 2: 보증 상태 응답

### 턴 3: 이슈 설명

### 턴 4: 해결책 제공

## 8. 상태 전환 시각화

각 턴에서 상태가 어떻게 변경되는지 확인합니다.

In [ ]:
# 전체 대화 흐름 요약

## 9. 보증 기간 만료 시나리오

보증 기간이 만료된 경우의 워크플로우를 테스트합니다.

In [ ]:
# 새로운 스레드로 보증 기간 만료 시나리오
# 턴 1
# 턴 2: 보증 기간 만료
# 턴 3: 하드웨어 문제
# 턴 4: 해결책 (에스컬레이션 예상)

## 주요 포인트 정리

1. **단일 에이전트, 다중 상태**: 하나의 에이전트가 여러 상태를 거침
2. **상태 기반 라우팅**: 현재 상태에 따라 다른 프롬프트와 도구 사용
3. **상태 전환**: 도구 호출을 통해 상태 업데이트 및 전환
4. **체크포인터**: 대화 이력과 상태를 유지
5. **wrap_model_call + request.override**: 상태에 따라 프롬프트와 도구를 단계별로 주입

**다음 단계**: 
- [330_Router_Pattern.py](330_Router_Pattern.py)에서 라우터 패턴 학습
- [340_Skills_Pattern.py](340_Skills_Pattern.py)에서 스킬 기반 패턴 학습